In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass
from typing import Any

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langchain_community.tools import DuckDuckGoSearchResults

In [3]:
@dataclass
class UserContext:
    user_id: str

In [4]:
store = InMemoryStore()

In [5]:
ddg_tool = DuckDuckGoSearchResults(output_format="list", max_results=8)
ddg_tool.invoke("최근 넷플릭스의 영화중 액션장르의 영화를 추천해줘")

[{'snippet': '멤버십에 넷플릭스 키즈 환경이 포함되어 있어 자녀가 자기만의 공간에서 가족용 시리즈와 영화를 즐기는 동안 부모가 이를 관리할 수 있습니다.',
  'title': '넷플릭스 대한민국 - 인터넷으로 시리즈와 영화를 시청하세요',
  'link': 'https://www.netflix.com/kr/'},
 {'snippet': "넷플릭스의 전성기는 끝났나? 2021년 '오징어 게임'은 넷플릭스 역사상 최다 시청률을 기록한 시리즈가 됐다. 사진 출처, Noh Juhan | Netflix.",
  'title': '넷플릭스의 전성기는 끝났나? - BBC News 코리아',
  'link': 'https://www.bbc.com/korean/international-61172759'},
 {'snippet': '최휘 : 최근 넷플릭스의 워너브라더스 인수 소식도 화제였잖아요.',
  'title': '[사회][열린라디오] 2026년 넷플릭스의 기대작은 어떤 작품인가? | YTN',
  'link': 'https://ytn.co.kr/_ln/0103_202601270059165889'},
 {'snippet': '',
  'title': '강원도 영화를 찍고 싶은데 장소 추천해줘!',
  'link': 'https://www.youtube.com/watch?v=PlOe6AG98RM'}]

In [6]:
@tool
def save_favorite_genre(genre: str, runtime: ToolRuntime[UserContext]) -> str:
    """
    사용자가 좋아하는 영화 장르를 장기 메모리에 저장합니다.
    예: genre='액션'
    """

    namespace = ("user", runtime.context.user_id, "preferences")

    runtime.store.put(namespace, "favorite_genre", {"genre": genre})

    return f"좋아하는 장르를 {genre}로 저장했습니다."

In [7]:
@tool
def load_favorite_genre(runtime: ToolRuntime[UserContext]) -> str:
    """저장된 영화 장르를 조회합니다."""

    namespace = ("user", runtime.context.user_id, "preferences")
    memory = runtime.store.get(namespace, "favorite_genre")

    if memory is None:
        return "선호하는 영화 장르가 없습니다."

    return f"저장된 선호 영화 장르는 {memory.value['genre']}입니다."

In [8]:
movie_recommender_agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    system_prompt=(
        "당신은 영화 추천글을 만들어주는 영화 추천가입니다. "
        "입력으로 사용자의 선호 장르와 최신 웹검색 결과가 주어집니다."
        "검색 결과의 영화들중 그 장르에 맞는 영화를 골라서 추천글을 작성해주세요"
        "반드시 아래 원칙을 따르세요"
        "1. 검색 결과에 근거해서 추천글을 작성하세요"
        "2. 검색결과에 없는 영화내용을 지어서 작성하지 마세요"
        "3. 각 영화마다 추천 이유를 간략하게 1문장으로 작성해주세요"
        "4. 추천글은 한국어로 작성하세요 "
    ),
)

In [9]:
@tool
def recommand_recent_movies(runtime: ToolRuntime[UserContext]) -> str:
    """
    저장된 종아하는 장르의 영화를 기반으로 덕덕고서치에서 최근 영화를 검색한뒤 추천글을 작성합니다.
    """
    namespace = ("user", runtime.context.user_id, "preferences")
    memory = runtime.store.get(namespace, "favorite_genre")

    if memory is None:
        return "종아하는 영화 장르가 저장되어 있지 않습니다. 먼저 좋아하는 영화 장르 정보를 알려주세요."

    genre = memory.value["genre"]

    search_query = f"넷플릭스 영화중 {genre} 장르의 영화 찾아줘"

    try:
        search_results = ddg_tool.invoke(search_query)
    except Exception as e:
        return f"덕덕고에서 검색을 실패하였습니다."

    print("덕덕고 결과 :", search_results)

    results = movie_recommender_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": f" ===== {search_results} ===== 위 정보를 활용하여 영화 추천글을 작성해주세요",
                }
            ]
        }
    )

    print("추천글 결과 : ", results["messages"][-1].content)
    return results["messages"][-1].content

In [10]:
agent = create_agent(
    model="gpt-5-mini",
    tools=[recommand_recent_movies, load_favorite_genre, save_favorite_genre],
    store=store,
    context_schema=UserContext,
    system_prompt=(
        "당신은 사용자의 영화의 취향을 기억하고 추천해주는 AI에이전트 입니다."
        "사용자가 좋아하는 영화 장르를 말하면 save_favorite_genre툴을 사용하여 장르를 저장하세요"
        "사용자가 저장된 영화 장르를 물어보면 load_favorite_genre툴을 사용하여 장르를 알려줍니다."
        "사용자가 영화 추천을 요청하면 recommand_recent_movies툴을 사용하여 영화를 추천합니다."
        "답변은 한국어로 해주세요"
    ),
)

In [11]:
user_context = UserContext(user_id="hanho")

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "나는 영화 액션 장르를 종아해 기억해줘"}
        ]
    },
    context=user_context,
)

result

{'messages': [HumanMessage(content='나는 영화 액션 장르를 종아해 기억해줘', additional_kwargs={}, response_metadata={}, id='3643e266-5be9-49d6-82b4-0a6806c8dd55'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 326, 'total_tokens': 417, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWd9UJ3BtFxLI1O36XLoUgZA997wZ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da9bf-1bf6-7440-aa77-253323716385-0', tool_calls=[{'name': 'save_favorite_genre', 'args': {'genre': '액션'}, 'id': 'call_K4vHPHGH6oHkmKAkaiztuaGp', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 326, 'output_tokens': 

In [12]:
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "내가 좋아하는 영화 장르가 뭐라고했지?"}
        ]
    },
    context=user_context,
)

result

{'messages': [HumanMessage(content='내가 좋아하는 영화 장르가 뭐라고했지?', additional_kwargs={}, response_metadata={}, id='181708dc-2699-460c-9d56-35af3053ed5c'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 150, 'prompt_tokens': 326, 'total_tokens': 476, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWd9qRTFyzNLw6CMMZOkqiuYITtjd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da9bf-759a-76c1-b8ba-6793fa6c2e07-0', tool_calls=[{'name': 'load_favorite_genre', 'args': {}, 'id': 'call_GuTzsQoFHCOhWLO2Tu7dnnpa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 326, 'output_tokens': 150, 'total

In [13]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "내가 좋아하는 영화 장르를 불러와서 비슷한 영화로 추천해줘",
            }
        ]
    },
    context=user_context,
)
result

덕덕고 결과 : [{'snippet': 'Jul 19, 2023 · 공상 과학 액션 시퀀스부터 예상치 못한 버디캅 코미디까지, 이 목록에는 현재 넷플릭스에서 볼 수 있는 최고의 액션 영화가 포함되어 있으며, 소파에 앉아 있는 모든 분께 만족감을 선사할 것입니다.', 'title': '[넷플릭스] 최신 액션 영화 추천 BEST 15 : 네이버 블로그', 'link': 'https://m.blog.naver.com/wkdckdgur34/223160791663'}, {'snippet': 'Dec 12, 2025 · 2026년 넷플릭스에서 지금 바로 볼 만한 인기 영화를 장르별·평점별로 추천합니다. 액션, 로맨스, 공포 등 취향에 딱 맞는 TOP 20 넷플릭스 영화를 발견해보세요!', 'title': '[2026 최신] 장르별로 넷플릭스 추천 영화 리스트', 'link': 'https://kr.fliflik.com/video-tips/best-netflix-movies-to-watch/'}, {'snippet': 'May 31, 2025 · 넷플릭스에서는 다양한 액션 장르의 베스트 영화들을 한 눈에 만나볼 수 있습니다. 최신 액션 영화 추천과 함께, 집에서 편안하게 즐길 수 있는 넷플릭스 추천작을 한데 모았습니다. 짜릿한 액션과 스리를 동시에 선사할 최고의 영화들을 확인해보세요.', 'title': '넷플릭스 액션 영화 추천 24편 - 지플릭스', 'link': 'https://gflix.kr/넷플릭스-액션-영화-추천-18편/'}, {'snippet': 'Nov 20, 2025 · 넷플릭스 최고 액션 영화 총정리! 숨겨진 명작부터 화제의 신작까지, 전문가가 엄선한 장르별 추천 리스트로 오늘 밤 스릴 넘치는 홈시어터를 즐겨보세요. 완전한 가이드를 확인하세요', 'title': '넷플릭스 액션 영화 추천｜장르별 명작·신작·평점 순위', 'link': 'https://smartprice.kr/넷플릭스-액션-영화-추천｜장르별-명작·신작·평점/'}]
추천글 결과 :  좋아요!

{'messages': [HumanMessage(content='내가 좋아하는 영화 장르를 불러와서 비슷한 영화로 추천해줘', additional_kwargs={}, response_metadata={}, id='c6e0e508-7102-47fd-bb29-312de19406d4'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 437, 'prompt_tokens': 333, 'total_tokens': 770, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWdAAhNnS8BxU4NzQ4upRsPyFGMpp', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da9bf-c33f-7de1-8670-711cf26489d0-0', tool_calls=[{'name': 'load_favorite_genre', 'args': {}, 'id': 'call_UP30JWOtqCD6FLFWOujtxmJj', 'type': 'tool_call'}, {'name': 'recommand_recent_movies', 'args': {}, 'id': 'call_nzFpCIK3nIfM45CbWv